In [1]:
from typing import List, Dict, Union, Any, Tuple
import numpy as np
import tensorflow as tf
import random
import datetime
import time
import json
import csv
from collections import deque
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
import secrets

In [2]:
class AC:
    def __init__(self,location:str,temperature:int):
        self.location = location
        self.temperature = temperature
    
    def cpntrol_ac(self,temperature:int):
        if 0<= temperature <= 30:
            self.temperature = temperature
        else:
            raise ValueError("Temperature should be between 0 and 30")

    def fetch_status(self)-> dict:
        return {"location":self.location,"temperature":self.temperature}

In [3]:
class AC:
    def __init__(self, location, temperature):
        self.location = location
        self.temperature = temperature

    def control_ac(self, temperature):
        if 18 <= temperature <= 30:
            self.temperature = temperature
            print(f"AC at {self.location} set to {temperature}°C")
        else:
            print(f"Invalid temperature range. Please set between 18°C and 30°C.")

    def fetch_status(self):
        status = {
            "location": self.location,
            "temperature": self.temperature,
            "power": "ON" if self.temperature != 0 else "OFF",
        }
        return status


class Aggregator:
    def __init__(self, time, motion, fan_status, *args, **kwargs):
        self.time = time
        self.motion = motion
        self.fan_status = fan_status

    def define_ac_objects(self, ac_info):
        self.ac_objects = [AC(ac["location"], ac["temperature"]) for ac in ac_info]

    def define_ai_object(self, ai_params):
        

    def invoke_ac_control(self, ac_obj, temperature):
        ac_obj.control_ac(temperature)

    def save_ai_output_to_csv(self, ai_output):

        print("Saving AI output to CSV not implemented yet.")

    def compute_difference(self, previous_state, current_state):
        difference = {}
        for key, value in current_state.items():
            if previous_state.get(key) != value:
                difference[key] = value
                if key == "temperature":
                    self.send_signal()  # Send signal if temperature changes manually
        return difference

    def compute_future_difference(self, current_state, next_predicted_state):
        difference = {}
        for key, value in next_predicted_state.items():
            if current_state.get(key) != value:
                difference[key] = value
        return difference

    def send_signal(self):
        # Replace with your actual signal sending logic
        print("Sending signal not implemented yet.")


# Example usage
ac_info = [
    {"location": "Living Room", "temperature": 22},
    {"location": "Bedroom", "temperature": 20},
]

aggregator = Aggregator(time="12:00", motion="inactive", fan_status="off")
aggregator.define_ac_objects(ac_info)

# Example AC control
aggregator.invoke_ac_control(aggregator.ac_objects[0], 24)

# Example AI integration (replace with your actual AI object and logic)
# ai_object = aggregator.define_ai_object(model_type="deep_learning")
# ai_output = ai_object.predict(...)  # Replace with your AI prediction logic
# aggregator.save_ai_output_to_csv(ai_output)

# Example state difference calculation
previous_state = {"temperature": 22}
current_state = {"temperature": 24}
difference = aggregator.compute_difference(previous_state, current_state)
print(f"Difference: {difference}")

# Example future difference calculation
next_predicted_state = {"temperature": 23}
future_difference = aggregator.compute_future_difference(current_state, next_predicted_state)
print(f"Future difference: {future_difference}")


AC at Living Room set to 24°C
Sending signal not implemented yet.
Difference: {'temperature': 24}
Future difference: {'temperature': 23}


In [6]:
import numpy as np
import tensorflow as tf
import datetime
import csv
from typing import List, Dict, Union, Any, Tuple

class AC:
    def __init__(self, location: str, temperature: int):
        self.location = location
        self.temperature = temperature

    def control_ac(self, temperature: int):
        if 18 <= temperature <= 30:
            self.temperature = temperature
            print(f"AC at {self.location} set to {temperature}°C")
        else:
            print(f"Invalid temperature range. Please set between 18°C and 30°C.")

    def fetch_status(self) -> dict:
        status = {
            "location": self.location,
            "temperature": self.temperature,
            "power": "ON" if self.temperature != 0 else "OFF",
        }
        return status


class AIAgent:
    def __init__(self, model_type: str):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(3,)),  
            tf.keras.layers.Dense(13, activation='softmax') 
        ])
        return model

    def train_rl(self, dataset: List[Dict[str, Any]]) -> None:
        # train PPO model
        states = np.array([[sample['location'], sample['temperature']] for sample in dataset])
        actions = np.array([sample['temperature'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['temperature']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions - 18, num_classes=13)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def predict(self, input_data: Dict[str, Any]) -> Tuple[int, float]:
        location_encoded = self.encode_location(input_data['location'])
        state = np.array([[location_encoded, input_data['temperature'],0]])
        #prediction
        policy_prob = self.policy(state, training=False).numpy()
        temperature = np.random.choice(np.arange(18, 31), p=policy_prob.flatten())

        return temperature, np.max(policy_prob)

    def define_advantage(self, temperature: int) -> float:
        return 1.0

    def encode_location(self, location: str) -> float:
    
        return float(hash(location))


class Aggregator:
    def __init__(self, *args, **kwargs):
        self.time = 0
        self.motion = False
        self.AC_temp_status = 0
        self.ac_objects = {}
        self.ai_object = None

    def define_ai_object(self, ai_params: dict) -> "AIAgent":
        self.ai_object = AIAgent(**ai_params)
        return self.ai_object

    def run(self, current_state: dict) -> List[dict]:
       
        self.time = current_state.get('time', 0)
        self.motion = current_state.get('motion', False)
        self.AC_temp_status = current_state.get('AC_temp_status', 0)
        for ac_obj in self.ac_objects.values():
            self.invoke_ac_control(ac_obj)
        statuses = [ac_obj.fetch_status() for ac_obj in self.ac_objects.values()]

        return statuses

    def invoke_ac_control(self, ac_obj: AC):
        temperature, _ = self.ai_object.predict({"location": ac_obj.location, "temperature": ac_obj.temperature})
        ac_obj.control_ac(temperature)

    def define_ac_objects(self, ac_info: List[Dict[str, Union[str, int]]]):
        for info in ac_info:
            location = info["location"]
            temperature = info["temperature"]
            self.ac_objects[location] = AC(location, temperature)

# Example 
if __name__ == "__main__":
    ac_info = [{"location": "Living Room", "temperature": 20}, {"location": "Bedroom", "temperature": 22}]
    aggregator = Aggregator()
    aggregator.define_ac_objects(ac_info)
    ai_params = {"model_type": "ReinforcementLearning"}
    aggregator.define_ai_object(ai_params)
    current_state = {"time": datetime.datetime.now(), "motion": True, "AC_temp_status": 0}
    statuses = aggregator.run(current_state)
    for status in statuses:
        print(status)


AC at Living Room set to 30°C
AC at Bedroom set to 30°C
{'location': 'Living Room', 'temperature': 30, 'power': 'ON'}
{'location': 'Bedroom', 'temperature': 30, 'power': 'ON'}


In [7]:
"""AC Class: This class represents an air conditioning unit. It has methods to control the temperature (control_ac) and fetch its status (fetch_status).

AIAgent Class: This class represents an intelligent agent responsible for predicting the temperature setting of the AC units. It has methods to build a policy network (build_policy_network), train the reinforcement learning model (train_rl), predict the temperature setting (predict), define advantage for training (define_advantage), and encode the location (encode_location).

Aggregator Class: This class serves as the central controller for the home automation system, focusing on AC control. It has methods to define the AI agent (define_ai_object), run the aggregation process (run), invoke AC control (invoke_ac_control), and define AC objects (define_ac_objects).

Now, let's explain the code in detail, function by function:

AC Class:

__init__(self, location: str, temperature: int): Initializes an instance of the AC class with a specified location and temperature.
control_ac(self, temperature: int): Controls the AC unit by setting the desired temperature within a valid range.
fetch_status(self) -> dict: Fetches the current status of the AC unit, including its location, temperature, and power status.
AIAgent Class:

__init__(self, model_type: str): Initializes an instance of the AIAgent class with the specified model type.
build_policy_network(self): Builds a policy network model using TensorFlow Keras Sequential API.
train_rl(self, dataset: List[Dict[str, Any]]) -> None: Trains the reinforcement learning model using the provided dataset.
predict(self, input_data: Dict[str, Any]) -> Tuple[int, float]: Predicts the temperature setting for the given input data (location and temperature).
define_advantage(self, temperature: int) -> float: Defines the advantage for training the reinforcement learning model.
encode_location(self, location: str) -> float: Encodes the location string into a numeric representation (dummy implementation).
Aggregator Class:

__init__(self, *args, **kwargs): Initializes an instance of the Aggregator class with default values for time, motion, AC_temp_status, and empty dictionaries for AC objects and AI agent.
define_ai_object(self, ai_params: dict) -> "AIAgent": Defines the AI agent with the specified parameters (model type).
run(self, current_state: dict) -> List[dict]: Runs the aggregation process by invoking AC control for each AC object, fetching their statuses, and returning a list of statuses.
invoke_ac_control(self, ac_obj: AC): Invokes AC control by predicting the temperature setting using the AI agent and controlling the AC object accordingly.
define_ac_objects(self, ac_info: List[Dict[str, Union[str, int]]]): Defines AC objects based on the provided information (location and initial temperature).
Finally, in the example usage:

An instance of the Aggregator class is created.
AC objects are defined with initial location and temperature information.
An AI agent is defined with the specified model type.
The current state is simulated with time, motion, and AC_temp_status information.
The aggregation process is run, and the statuses of AC objects are printed."""

'AC Class: This class represents an air conditioning unit. It has methods to control the temperature (control_ac) and fetch its status (fetch_status).\n\nAIAgent Class: This class represents an intelligent agent responsible for predicting the temperature setting of the AC units. It has methods to build a policy network (build_policy_network), train the reinforcement learning model (train_rl), predict the temperature setting (predict), define advantage for training (define_advantage), and encode the location (encode_location).\n\nAggregator Class: This class serves as the central controller for the home automation system, focusing on AC control. It has methods to define the AI agent (define_ai_object), run the aggregation process (run), invoke AC control (invoke_ac_control), and define AC objects (define_ac_objects).\n\nNow, let\'s explain the code in detail, function by function:\n\nAC Class:\n\n__init__(self, location: str, temperature: int): Initializes an instance of the AC class wi

In [8]:
    """
    Certainly! Let's go into detail about the `AIAgent` class and its `train_rl` method:

### AIAgent Class

The `AIAgent` class is responsible for managing the AI model used for predicting the temperature settings of the air conditioning units. Here's a breakdown of its key components:

#### Initialization (`__init__` method):

```python
def __init__(self, model_type: str):
    self.model_type = model_type

    if model_type == 'ReinforcementLearning':
        self.policy = self.build_policy_network()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
```

- The `__init__` method initializes an instance of the `AIAgent` class with a specified `model_type`.
- It checks if the model type is "ReinforcementLearning" and if so, it builds the policy network for reinforcement learning and initializes an Adam optimizer for training the model.

#### Building Policy Network (`build_policy_network` method):

```python
def build_policy_network(self):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(3,)),  
        tf.keras.layers.Dense(13, activation='softmax') 
    ])
    return model
```

- The `build_policy_network` method constructs a neural network model using TensorFlow's Keras Sequential API.
- The model consists of two fully connected layers (`Dense` layers) with ReLU activation for the first layer and softmax activation for the output layer.
- The input shape is defined as `(3,)`, indicating that the input features include the encoded location, temperature, and an additional dummy feature.

#### Training the Reinforcement Learning Model (`train_rl` method):

```python
def train_rl(self, dataset: List[Dict[str, Any]]) -> None:
    # Train the PPO model
    states = np.array([[sample['location'], sample['temperature']] for sample in dataset])
    actions = np.array([sample['temperature'] for sample in dataset])
    advantages = np.array([self.define_advantage(sample['temperature']) for sample in dataset])

    actions_categorical = tf.keras.utils.to_categorical(actions - 18, num_classes=13)

    with tf.GradientTape() as tape:
        policy_prob = self.policy(states, training=True)
        chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
        old_policy_prob = tf.stop_gradient(chosen_prob)

        ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
        clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
        surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
        loss = tf.reduce_mean(surrogate)

    gradients = tape.gradient(loss, self.policy.trainable_variables)
    self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))
```

- The `train_rl` method is responsible for training the reinforcement learning model using the Proximal Policy Optimization (PPO) algorithm.
- It takes a dataset of samples as input, where each sample contains the location and temperature information.
- The method calculates advantages, which represent the benefits of taking certain actions (temperature settings) over others.
- It converts the actions (temperature settings) into categorical form for training the model.
- The training process is wrapped in a TensorFlow GradientTape to compute gradients for the policy network.
- It computes the loss function based on the PPO objective.
- Finally, it applies gradient updates to the policy network parameters using the Adam optimizer.

In summary, the `AIAgent` class provides functionality for initializing, building, training, and using an AI model (specifically for reinforcement learning) to predict the temperature settings of the air conditioning units based on their locations and current temperatures.


Let's break down the training process step by step:

1. **GradientTape Context Manager**:
   - `with tf.GradientTape() as tape:`: This context manager is used to record operations for automatic differentiation. Inside this block, TensorFlow keeps track of all operations applied to the trainable variables, allowing us to compute their gradients with respect to a loss function.

2. **Forward Pass**:
   - `policy_prob = self.policy(states, training=True)`: This line performs a forward pass through the policy network. Given the input states (locations and temperatures), it computes the probability distribution over the possible actions (temperature settings). The `training=True` argument indicates that the model is being used for training, which can enable certain operations like dropout.

3. **Compute Chosen Probabilities**:
   - `chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)`: Here, we compute the probabilities of the chosen actions. `actions_categorical` contains the one-hot encoded actions (temperature settings), and `policy_prob` contains the predicted probabilities by the policy network. Multiplying them element-wise and summing along the action dimension gives the probabilities of the chosen actions.

4. **Compute Old Policy Probabilities**:
   - `old_policy_prob = tf.stop_gradient(chosen_prob)`: We compute the old policy probabilities, which are the probabilities of the chosen actions from the previous iteration. We use `tf.stop_gradient` to prevent these probabilities from contributing to the gradient computation, as they are considered fixed during optimization.

5. **Compute Ratios**:
   - `ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))`: This line computes the ratios between the new and old policy probabilities. These ratios are essential for computing the surrogate loss, which guides the policy updates. We add a small epsilon (`1e-10`) to avoid numerical instability when taking logarithms.

6. **Clip Ratios**:
   - `clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)`: Here, we clip the ratios to a certain range (`[1 - 0.2, 1 + 0.2]`). This is a common practice in policy gradient methods like PPO to prevent large policy updates that could destabilize training.

7. **Compute Surrogate Loss**:
   - `surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)`: This line computes the surrogate loss, which is a form of policy gradient loss used in PPO. It balances the benefits of taking certain actions (given by `advantages`) with the policy update step size (given by `clipped_ratio`).

8. **Compute Mean Loss**:
   - `loss = tf.reduce_mean(surrogate)`: The surrogate loss is then averaged over the batch to obtain a scalar loss value, which represents the overall performance of the policy on the given batch of data.

9. **Backward Pass**:
   - `gradients = tape.gradient(loss, self.policy.trainable_variables)`: After computing the loss, we use the gradient tape to compute the gradients of the loss with respect to the trainable variables of the policy network.

10. **Apply Gradients**:
   - `self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))`: Finally, we apply the computed gradients to update the parameters (trainable variables) of the policy network using an optimizer (in this case, Adam optimizer).

In summary, this code segment implements the training loop for the reinforcement learning model using the Proximal Policy Optimization (PPO) algorithm. It calculates the surrogate loss, computes gradients, and updates the policy network parameters to improve its performance in predicting temperature settings for the air conditioning units.
    
    """

'\nCertainly! Let\'s go into detail about the `AIAgent` class and its `train_rl` method:\n\n### AIAgent Class\n\nThe `AIAgent` class is responsible for managing the AI model used for predicting the temperature settings of the air conditioning units. Here\'s a breakdown of its key components:\n\n#### Initialization (`__init__` method):\n\n```python\ndef __init__(self, model_type: str):\nself.model_type = model_type\n\nif model_type == \'ReinforcementLearning\':\n    self.policy = self.build_policy_network()\n    self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)\n```\n\n- The `__init__` method initializes an instance of the `AIAgent` class with a specified `model_type`.\n- It checks if the model type is "ReinforcementLearning" and if so, it builds the policy network for reinforcement learning and initializes an Adam optimizer for training the model.\n\n#### Building Policy Network (`build_policy_network` method):\n\n```python\ndef build_policy_network(self):\nmodel = tf.kera